In [1]:
from pyspark.sql import SparkSession

In [2]:
spark =    SparkSession.builder \
    .appName("day11") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access",  "true") \
    .config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.catalogImplementation",        "hive") \
    .config("hive.metastore.uris",                    "thrift://metastore:9083") \
    .config("spark.eventLog.enabled",                 "true") \
    .config("spark.eventLog.dir",                     "s3a://spark-event/") \
    .config("spark.sql.shuffle.partitions",           str(4)) \
    .config("spark.sql.adaptive.enabled",             "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
spark.sparkContext.setLogLevel("WARN")

transformation

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
# ── HIVE CHECK ────────────────────────────────────────────────────────────────
# local_db is auto-created by init-hive-db container on first startup
print("\n── Hive databases (local_db should appear) ────────────────────────────")
spark.sql("SHOW DATABASES").show()


── Hive databases (local_db should appear) ────────────────────────────
+---------+
|namespace|
+---------+
|  default|
| local_db|
+---------+



In [6]:
spark.sql("SHOW TABLES IN local_db").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
| local_db|  sample_hive_table|      false|
| local_db|total_amount_ranked|      false|
+---------+-------------------+-----------+



In [7]:
try:
    df = spark.sql("SELECT * FROM local_db.sample_hive_table")
    print("[INFO] Loaded from Hive: local_db.sample_hive_table")
except Exception:
    df = spark.read.parquet("s3a://spark-warehouse/raw_data/save_from_notebook")
    print("[WARN] Hive table not found — loaded from Parquet. Run Day 9 first.")
 
taxi_zone = spark.read \
    .option("header", "true").option("inferSchema", "true") \
    .csv("s3a://metadata-rawdata/taxi_zone_lookup.csv")
 
 
print(f"taxi : {df.count()} rows")
print(f"taxi_zone : {taxi_zone.count()} rows")

[INFO] Loaded from Hive: local_db.sample_hive_table
taxi : 300000 rows
taxi_zone : 265 rows


In [8]:
# ── SECTION 1: REGISTERING VIEWS ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 1 — Registering Views")
print("=" * 60)


SECTION 1 — Registering Views


In [9]:
# Temp view — lives for this SparkSession only, gone when spark.stop()
df.createOrReplaceTempView("taxi")
taxi_zone.createOrReplaceTempView("zone")

In [10]:
# Global temp view — visible across sessions via global_temp prefix
taxi_zone.createOrReplaceGlobalTempView("zone_global")

In [11]:
# Both produce identical results:
print("── DataFrame API ──────────────────────────────────────")
df.groupBy("tpep_pickup_date").count().orderBy("tpep_pickup_date").show()
 
print("── Equivalent SQL ─────────────────────────────────────")
spark.sql("""
    SELECT tpep_pickup_date, COUNT(tpep_pickup_date) AS count
    FROM taxi
    GROUP BY tpep_pickup_date
    ORDER BY tpep_pickup_date
""").show()

── DataFrame API ──────────────────────────────────────
+----------------+-----+
|tpep_pickup_date|count|
+----------------+-----+
|      2025-01-01|90188|
|      2025-01-02|84832|
|      2025-01-03|91250|
|      2025-01-04|33730|
+----------------+-----+

── Equivalent SQL ─────────────────────────────────────
+----------------+-----+
|tpep_pickup_date|count|
+----------------+-----+
|      2025-01-01|90188|
|      2025-01-02|84832|
|      2025-01-03|91250|
|      2025-01-04|33730|
+----------------+-----+



In [12]:
# Global view requires 'global_temp.' prefix
print("── Global temp view (cross-session) ───────────────────")
spark.sql("SELECT COUNT(*) AS total FROM global_temp.zone_global").show()

── Global temp view (cross-session) ───────────────────
+-----+
|total|
+-----+
|  265|
+-----+



In [13]:
# ── SECTION 2: FULL SQL FEATURE SET ──────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 2 — SQL Feature Set")
print("=" * 60)


SECTION 2 — SQL Feature Set


In [15]:
# 2a. JOIN in SQL — same broadcast hint available
print("── SQL JOIN with broadcast hint ───────────────────────")
spark.sql("""
    SELECT
        t.tpep_pickup_date,
        z.Borough,
        z.Zone,
        t.total_amount
    FROM taxi t
    LEFT JOIN zone z ON t.PULocationID = z.LocationID
    ORDER BY t.tpep_pickup_date, t.total_amount DESC
    LIMIT 15
""").show()

── SQL JOIN with broadcast hint ───────────────────────
+----------------+---------+--------------------+------------+
|tpep_pickup_date|  Borough|                Zone|total_amount|
+----------------+---------+--------------------+------------+
|      2025-01-01|   Queens|         JFK Airport|      865.39|
|      2025-01-01|   Queens|           Glen Oaks|      794.82|
|      2025-01-01|      N/A|      Outside of NYC|       701.0|
|      2025-01-01|      N/A|      Outside of NYC|       701.0|
|      2025-01-01|Manhattan|   East Harlem North|      525.36|
|      2025-01-01|   Queens|         JFK Airport|      516.13|
|      2025-01-01|Manhattan|        Clinton East|       478.5|
|      2025-01-01|      N/A|      Outside of NYC|       451.0|
|      2025-01-01|   Queens|         JFK Airport|      423.33|
|      2025-01-01|Manhattan|        Clinton East|      420.38|
|      2025-01-01|Manhattan|Penn Station/Madi...|       406.9|
|      2025-01-01|   Queens|   LaGuardia Airport|      405.72|

In [17]:
# 2b. CASE WHEN — conditional column
print("── CASE WHEN  ───────────────────────────────────")
spark.sql("""
    SELECT
        tpep_pickup_date,
        PULocationID,
        total_amount,
        CASE
            WHEN VendorID = 1 THEN 'Creative Mobile Technologies LLC'
            WHEN VendorID = 2 THEN 'Curb Mobility LLC'
            WHEN VendorID = 6 THEN 'Myle Technologies Inc'
            WHEN VendorID = 7 THEN 'Helix'
            ELSE 'Unknown'
        END AS Vendor
    FROM taxi
    ORDER BY tpep_pickup_date, total_amount DESC
    LIMIT 15
""").show()

── CASE WHEN  ───────────────────────────────────
+----------------+------------+------------+--------------------+
|tpep_pickup_date|PULocationID|total_amount|              Vendor|
+----------------+------------+------------+--------------------+
|      2025-01-01|         132|      865.39|   Curb Mobility LLC|
|      2025-01-01|         101|      794.82|   Curb Mobility LLC|
|      2025-01-01|         265|       701.0|   Curb Mobility LLC|
|      2025-01-01|         265|       701.0|   Curb Mobility LLC|
|      2025-01-01|          74|      525.36|   Curb Mobility LLC|
|      2025-01-01|         132|      516.13|   Curb Mobility LLC|
|      2025-01-01|          48|       478.5|   Curb Mobility LLC|
|      2025-01-01|         265|       451.0|Creative Mobile T...|
|      2025-01-01|         132|      423.33|   Curb Mobility LLC|
|      2025-01-01|          48|      420.38|   Curb Mobility LLC|
|      2025-01-01|         186|       406.9|   Curb Mobility LLC|
|      2025-01-01|        

In [18]:
# 2c. Subquery — employees earning above company average
print("── Subquery  ────────────────────────────────────────")
spark.sql("""
    SELECT tpep_pickup_datetime, PULocationID, total_amount
    FROM taxi
    WHERE total_amount > (SELECT AVG(total_amount) FROM taxi WHERE total_amount > 0)
    ORDER BY total_amount DESC
    LIMIT 10
""").show()

── Subquery  ────────────────────────────────────────
+--------------------+------------+------------+
|tpep_pickup_datetime|PULocationID|total_amount|
+--------------------+------------+------------+
| 2025-01-01 23:13:43|         132|      865.39|
| 2025-01-01 12:53:37|         101|      794.82|
| 2025-01-03 06:30:10|         132|      702.11|
| 2025-01-01 13:47:30|         265|       701.0|
| 2025-01-01 13:56:20|         265|       701.0|
| 2025-01-03 18:45:03|         132|      607.01|
| 2025-01-01 10:38:17|          74|      525.36|
| 2025-01-01 10:05:04|         132|      516.13|
| 2025-01-02 16:13:46|         132|      480.26|
| 2025-01-01 06:46:05|          48|       478.5|
+--------------------+------------+------------+



In [21]:
print("── Test  ────────────────────────────────────────")
spark.sql("""
    SELECT tpep_pickup_datetime, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, total_amount, congestion_surcharge, airport_fee
    FROM taxi
    WHERE total_amount >0 and fare_amount <0
    ORDER BY total_amount DESC
""").show()

── Test  ────────────────────────────────────────
+--------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------+-----------+
|tpep_pickup_datetime|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|total_amount|congestion_surcharge|airport_fee|
+--------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------+-----------+
| 2025-01-01 03:42:26|          17|          43|           0|      -4.24|  0.0|    0.5|       0.0|       25.93|                NULL|       NULL|
| 2025-01-01 01:32:37|         226|          63|           0|      -8.78|  0.0|    0.5|       0.0|       25.11|                NULL|       NULL|
| 2025-01-01 04:56:49|         114|         116|           0|      -0.83|  0.0|    0.5|       0.0|       23.17|                NULL|       NULL|
| 2025-01-01 17:15:52|         129|          50|           0|      -0.62|  0.0| 

In [24]:
# 2d. CTEs — same as PostgreSQL, composes well for multi-step analytics
print("── CTE (dept stats + salary band) ─────────────────────")
spark.sql("""
    WITH filter_bad_data AS (
        SELECT *
        FROM taxi
        WHERE fare_amount >0 and total_amount >0
    ),
    date_zone_stats AS (
        SELECT
            tpep_pickup_date, 
            PULocationID,
            COUNT(*)                        AS transaction,
            ROUND(AVG(fare_amount), 2)      AS avg_fare,
            ROUND(AVG(total_amount), 2)     AS avg_total,
            ROUND(SUM(total_amount), 2)     AS sum_total_amount
        FROM filter_bad_data
        GROUP BY tpep_pickup_date, PULocationID
    ),
    date_zone_ranked AS (
        SELECT *,
            CASE
                WHEN avg_total >= 500 THEN 'high'
                WHEN avg_total >= 100 THEN 'mid'
                ELSE 'low'
            END AS avg_total_band,
            RANK() OVER (ORDER BY avg_total DESC) AS avg_total_rank
        FROM date_zone_stats
    )
    SELECT * FROM date_zone_ranked ORDER BY avg_total_rank
""").show()

── CTE (dept stats + salary band) ─────────────────────
+----------------+------------+-----------+--------+---------+----------------+--------------+--------------+
|tpep_pickup_date|PULocationID|transaction|avg_fare|avg_total|sum_total_amount|avg_total_band|avg_total_rank|
+----------------+------------+-----------+--------+---------+----------------+--------------+--------------+
|      2025-01-01|         101|          4|  195.79|   216.95|          867.78|           mid|             1|
|      2025-01-03|         265|         43|  103.83|   116.03|         4989.28|           mid|             2|
|      2025-01-01|         265|        106|  105.99|   113.87|        12070.66|           mid|             3|
|      2025-01-04|         245|          1|   91.88|   111.46|          111.46|           mid|             4|
|      2025-01-04|         202|          1|   80.46|    97.75|           97.75|           low|             5|
|      2025-01-04|         194|          1|    87.0|    94.94|  

In [27]:
# 2e. Window functions in SQL — identical to PostgreSQL syntax
print("── Window functions in SQL ─────────────────────────────")
spark.sql("""
    SELECT
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        PULocationID,
        total_amount,
        RANK()      OVER (PARTITION BY PULocationID ORDER BY total_amount DESC) AS pickup_location_rank,
        AVG(total_amount) OVER (PARTITION BY PULocationID)                      AS total_amount_avg,
        ROUND(total_amount / SUM(total_amount) OVER (PARTITION BY PULocationID) * 100, 1) AS pct_of_PULocation,
        LAG(total_amount) OVER (PARTITION BY PULocationID ORDER BY tpep_pickup_datetime)   AS prev_transaction
    FROM taxi
    WHERE fare_amount >0 and total_amount >0
    ORDER BY PULocationID, pickup_location_rank
    LIMIT 20
""").show()

── Window functions in SQL ─────────────────────────────
+--------+--------------------+---------------------+------------+------------+--------------------+-----------------+-----------------+----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|total_amount|pickup_location_rank| total_amount_avg|pct_of_PULocation|prev_transaction|
+--------+--------------------+---------------------+------------+------------+--------------------+-----------------+-----------------+----------------+
|       2| 2025-01-02 13:40:31|  2025-01-02 13:40:36|           1|       212.0|                   1|86.77869047619046|              2.9|            41.0|
|       1| 2025-01-03 17:09:23|  2025-01-03 17:09:58|           1|      206.62|                   2|86.77869047619046|              2.8|            90.0|
|       2| 2025-01-01 09:49:09|  2025-01-01 09:49:11|           1|      168.58|                   3|86.77869047619046|              2.3|          140.48|
|       2| 2025-01-

In [31]:
# 2f. Aggregation on the sales view
print("── Sales CTE: monthly totals + running total ───────────")
spark.sql("""
    WITH daily AS (
        SELECT
            tpep_pickup_date,
            PULocationID,
            total_amount,
            SUM(total_amount) OVER (
                PARTITION BY tpep_pickup_date, PULocationID
                ORDER BY tpep_pickup_datetime
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS accumulating_total,
            AVG(total_amount) OVER (
                PARTITION BY tpep_pickup_date, PULocationID
                ORDER BY tpep_pickup_datetime
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS rolling_3d_avg
        FROM taxi
        WHERE fare_amount >0 and total_amount >0
    )
    SELECT * FROM daily ORDER BY tpep_pickup_date, PULocationID
    LIMIT 20
""").show()

── Sales CTE: monthly totals + running total ───────────
+----------------+------------+------------+------------------+------------------+
|tpep_pickup_date|PULocationID|total_amount|accumulating_total|    rolling_3d_avg|
+----------------+------------+------------+------------------+------------------+
|      2025-01-01|           1|       115.0|             115.0|             115.0|
|      2025-01-01|           1|      110.98|225.98000000000002|112.99000000000001|
|      2025-01-01|           1|      140.48|366.46000000000004|122.15333333333335|
|      2025-01-01|           1|      168.58| 535.0400000000001|140.01333333333332|
|      2025-01-01|           1|       110.0| 645.0400000000001|139.68666666666667|
|      2025-01-01|           1|        73.2| 718.2400000000001|            117.26|
|      2025-01-01|           1|       139.2|            857.44|107.46666666666665|
|      2025-01-01|           1|        38.0|            895.44| 83.46666666666665|
|      2025-01-01|           1

In [30]:
# ── SECTION 3: MIXING SQL AND DATAFRAME API ───────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 3 — Mixing SQL and DataFrame API")
print("=" * 60)


SECTION 3 — Mixing SQL and DataFrame API


In [36]:
# Start with SQL for the readable aggregation step...
date_summary = spark.sql("""
    SELECT
        t.tpep_pickup_date,
        t.PULocationID,
        COUNT(t.tpep_pickup_datetime)     AS transaction,
        ROUND(AVG(t.total_amount), 2)     AS avg_total,
        ROUND(SUM(t.total_amount), 2)     AS sum_total
    FROM taxi t
    LEFT JOIN zone z ON t.PULocationID = z.LocationID
    WHERE t.fare_amount >0 and t.total_amount >0
    GROUP BY t.tpep_pickup_date, t.PULocationID
""")

In [39]:
# ...then switch to DataFrame API for programmatic column manipulation
final = date_summary \
    .withColumn("total_amount_band", F.when(F.col("avg_total") >= 200, "high")
                                 .when(F.col("avg_total") >= 50, "mid")
                                 .otherwise("low")) \
    .withColumn("cost_per_head", F.round(F.col("sum_total") / F.col("transaction"), 2)) \
    .orderBy(F.col("avg_total").desc())
 
print("── SQL aggregation → DataFrame enrichment ──────────────")
final.show()

── SQL aggregation → DataFrame enrichment ──────────────
+----------------+------------+-----------+---------+---------+-----------------+-------------+
|tpep_pickup_date|PULocationID|transaction|avg_total|sum_total|total_amount_band|cost_per_head|
+----------------+------------+-----------+---------+---------+-----------------+-------------+
|      2025-01-01|         101|          4|   216.95|   867.78|             high|       216.95|
|      2025-01-03|         265|         43|   116.03|  4989.28|              mid|       116.03|
|      2025-01-01|         265|        106|   113.87| 12070.66|              mid|       113.87|
|      2025-01-04|         245|          1|   111.46|   111.46|              mid|       111.46|
|      2025-01-04|         202|          1|    97.75|    97.75|              mid|        97.75|
|      2025-01-04|         194|          1|    94.94|    94.94|              mid|        94.94|
|      2025-01-02|           1|         28|     93.8|  2626.47|              mi

In [40]:
# Register the result and query again with SQL
final.createOrReplaceTempView("date_summary")
spark.sql("""
    SELECT total_amount_band, COUNT(*) AS zone, ROUND(AVG(avg_total), 2) AS band_avg
    FROM date_summary
    GROUP BY total_amount_band
    ORDER BY band_avg DESC
""").show()

+-----------------+----+--------+
|total_amount_band|zone|band_avg|
+-----------------+----+--------+
|             high|   1|  216.95|
|              mid| 116|   68.78|
|              low| 779|   28.85|
+-----------------+----+--------+



In [41]:
# ── SECTION 4: WHEN TO USE SQL VS DATAFRAME API ───────────────────────────────
print("\n" + "=" * 60)
print("SECTION 4 — SQL vs DataFrame API decision")
print("=" * 60)


SECTION 4 — SQL vs DataFrame API decision


In [42]:
# Both of these produce IDENTICAL physical plans — run .explain() to verify
print("── Verify identical plans ──────────────────────────────")

── Verify identical plans ──────────────────────────────


In [43]:
sql_plan = spark.sql("""
    SELECT tpep_pickup_date, ROUND(AVG(total_amount), 2) AS avg_amount
    FROM taxi WHERE fare_amount >0 and total_amount >0
    GROUP BY tpep_pickup_date
""")
 
api_plan = df \
    .filter((F.col("fare_amount") >0 ) & (F.col("total_amount") >0) ) \
    .groupBy("tpep_pickup_date") \
    .agg(F.round(F.avg("total_amount"), 2).alias("avg_amount"))
 
print("SQL plan:")
sql_plan.explain()
print("DataFrame API plan:")
api_plan.explain()
# Plans should be structurally identical

SQL plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[tpep_pickup_date#61], functions=[avg(total_amount#55)])
   +- Exchange hashpartitioning(tpep_pickup_date#61, 4), ENSURE_REQUIREMENTS, [plan_id=2167]
      +- HashAggregate(keys=[tpep_pickup_date#61], functions=[partial_avg(total_amount#55)])
         +- Project [total_amount#55, tpep_pickup_date#61]
            +- Filter (((isnotnull(fare_amount#50) AND isnotnull(total_amount#55)) AND (fare_amount#50 > 0.0)) AND (total_amount#55 > 0.0))
               +- FileScan parquet spark_catalog.local_db.sample_hive_table[fare_amount#50,total_amount#55,tpep_pickup_date#61] Batched: true, DataFilters: [isnotnull(fare_amount#50), isnotnull(total_amount#55), (fare_amount#50 > 0.0), (total_amount#55 ..., Format: Parquet, Location: CatalogFileIndex(1 paths)[s3a://spark-warehouse/hive/local_db/sample_hive_table], PartitionFilters: [], PushedFilters: [IsNotNull(fare_amount), IsNotNull(total_amount), GreaterThan(fare

In [44]:
# ── WRITE SQL OUTPUT AS HIVE TABLE ────────────────────────────────────────────
spark.sql("DROP TABLE IF EXISTS local_db.date_summary")
final.write.mode("overwrite").format("parquet").saveAsTable("local_db.date_summary")
print("\n✓ local_db.date_summary saved to Hive")
 
spark.sql("SHOW TABLES IN local_db").show()


✓ local_db.date_summary saved to Hive
+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
| local_db|       date_summary|      false|
| local_db|  sample_hive_table|      false|
| local_db|total_amount_ranked|      false|
|         |       date_summary|       true|
|         |               taxi|       true|
|         |               zone|       true|
+---------+-------------------+-----------+



**exercise1**    
temp table only persists in current spark session unitl running spark.stop()   
- global temp table can be shared across spark sessions but still disappear after spark.stop()    

save as permanent table in hive, making table persistent despite spark.stop()    


**exercise2**

In [46]:
spark.sql("""
    SELECT 
        tpep_pickup_date,
        COUNT(VendorID) as Vendor_count,
        ROUND( AVG(fare_amount), 2) as avg_fare,
        MAX( fare_amount) as max_fare,
        MIN( fare_amount) as min_fare,
        ROUND( SUM( fare_amount), 2) as total_fare
    FROM taxi
    GROUP BY tpep_pickup_date

""").show()

+----------------+------------+--------+--------+--------+----------+
|tpep_pickup_date|Vendor_count|avg_fare|max_fare|min_fare|total_fare|
+----------------+------------+--------+--------+--------+----------+
|      2025-01-01|       90188|   17.66|   826.2|  -826.2|1593061.51|
|      2025-01-03|       91250|   18.17|   670.1|  -175.2| 1657619.5|
|      2025-01-02|       84832|    19.1|   423.7|  -336.2|1620005.97|
|      2025-01-04|       33730|   17.67|   325.0|  -136.0| 596168.73|
+----------------+------------+--------+--------+--------+----------+



**explain3**    
both plans can work together as they are the same

In [45]:
print("SQL plan:")
sql_plan.explain(extended=True)
print("DataFrame API plan:")
api_plan.explain(extended=True)

SQL plan:
== Parsed Logical Plan ==
'Aggregate ['tpep_pickup_date], ['tpep_pickup_date, 'ROUND('AVG('total_amount), 2) AS avg_amount#1098]
+- 'Filter (('fare_amount > 0) AND ('total_amount > 0))
   +- 'UnresolvedRelation [taxi], [], false

== Analyzed Logical Plan ==
tpep_pickup_date: date, avg_amount: double
Aggregate [tpep_pickup_date#61], [tpep_pickup_date#61, round(avg(total_amount#55), 2) AS avg_amount#1098]
+- Filter ((fare_amount#50 > cast(0 as double)) AND (total_amount#55 > cast(0 as double)))
   +- SubqueryAlias taxi
      +- View (`taxi`, [VendorID#40,tpep_pickup_datetime#41,tpep_dropoff_datetime#42,passenger_count#43L,trip_distance#44,RatecodeID#45L,store_and_fwd_flag#46,PULocationID#47,DOLocationID#48,payment_type#49L,fare_amount#50,extra#51,mta_tax#52,tip_amount#53,improvement_surcharge#54,total_amount#55,congestion_surcharge#56,Airport_fee#57,cbd_congestion_fee#58,total_amount_thb#59,is_valid_passenger_count#60,tpep_pickup_date#61])
         +- Project [VendorID#40, tpep

**exercise4**

In [53]:
spark.sql("""
    SELECT 
        t.tpep_pickup_date,
        t.PULocationID, 
        z.BOROUGH,
        z.ZONE,
        t.total_amount,
        MAX( t.total_amount) OVER (PARTITION BY t.tpep_pickup_date, t.PULocationID ORDER BY t.tpep_pickup_datetime) AS top_earner, 
        MIN( t.total_amount) OVER (PARTITION BY t.tpep_pickup_date, t.PULocationID ORDER BY t.tpep_pickup_datetime) AS bottom_earner,
        ROUND(AVG( t.total_amount) OVER (PARTITION BY t.tpep_pickup_date, t.PULocationID ORDER BY t.tpep_pickup_datetime), 2) AS avg_amount,
        ROUND(t.total_amount / SUM( t.total_amount) OVER (PARTITION BY t.tpep_pickup_date, t.PULocationID ORDER BY t.tpep_pickup_datetime), 2) as pct_amount
    FROM taxi t 
    LEFT JOIN zone z ON t.PULocationID = z.LocationID
    WHERE t.fare_amount >0 and t.total_amount >0
    ORDER BY t.tpep_pickup_datetime
""").show()

+----------------+------------+---------+--------------------+------------+----------+-------------+----------+----------+
|tpep_pickup_date|PULocationID|  BOROUGH|                ZONE|total_amount|top_earner|bottom_earner|avg_amount|pct_amount|
+----------------+------------+---------+--------------------+------------+----------+-------------+----------+----------+
|      2025-01-01|         211|Manhattan|                SoHo|       74.05|     74.05|        74.05|     74.05|       1.0|
|      2025-01-01|         237|Manhattan|Upper East Side S...|        16.4|      16.4|         16.4|      16.4|       1.0|
|      2025-01-01|         107|Manhattan|            Gramercy|       17.85|     17.85|        17.85|     17.85|       1.0|
|      2025-01-01|         148|Manhattan|     Lower East Side|        12.9|      12.9|         12.9|      12.9|       1.0|
|      2025-01-01|         132|   Queens|         JFK Airport|        25.1|      25.1|         25.1|      25.1|       1.0|
|      2025-01-0

In [59]:
temp_df = df.join(
    F.broadcast(taxi_zone),
    df["PULocationID"] == taxi_zone["LocationID"],
    how="left"
)
df_window = Window.partitionBy("tpep_pickup_date", "PULocationID" ).orderBy("tpep_pickup_datetime")

temp_df.where((F.col("fare_amount") >0) & (F.col("total_amount") > 0)) \
    .withColumn("top_earner",  F.round(F.max("total_amount").over(df_window), 2)) \
    .withColumn("bottom_earner",  F.round(F.min("total_amount").over(df_window), 2)) \
    .withColumn("avg_amount",  F.round(F.avg("total_amount").over(df_window), 2)) \
    .withColumn("pct_amount",  F.round( F.col("total_amount")/ F.sum("total_amount").over(df_window), 2)) \
    .orderBy("tpep_pickup_datetime") \
    .select("tpep_pickup_date", "PULocationID", "BOROUGH", "ZONE", "total_amount", "top_earner", "bottom_earner", "avg_amount", "pct_amount").show() 

+----------------+------------+---------+--------------------+------------+----------+-------------+----------+----------+
|tpep_pickup_date|PULocationID|  BOROUGH|                ZONE|total_amount|top_earner|bottom_earner|avg_amount|pct_amount|
+----------------+------------+---------+--------------------+------------+----------+-------------+----------+----------+
|      2025-01-01|         211|Manhattan|                SoHo|       74.05|     74.05|        74.05|     74.05|       1.0|
|      2025-01-01|         237|Manhattan|Upper East Side S...|        16.4|      16.4|         16.4|      16.4|       1.0|
|      2025-01-01|         107|Manhattan|            Gramercy|       17.85|     17.85|        17.85|     17.85|       1.0|
|      2025-01-01|         148|Manhattan|     Lower East Side|        12.9|      12.9|         12.9|      12.9|       1.0|
|      2025-01-01|         132|   Queens|         JFK Airport|        25.1|      25.1|         25.1|      25.1|       1.0|
|      2025-01-0

In [60]:
spark.stop()
print("Done — local_db.date_summary saved, check :18080 for history")

Done — local_db.date_summary saved, check :18080 for history
